# Task 3: Python Data Processing & Analysis — NorthStar Urban Mobility

## Overview

This notebook uses Python (pandas, NumPy, matplotlib, seaborn) to clean, process, and analyse NorthStar's operational dataset. The analytical methods applied go beyond descriptive summary to identify patterns, anomalies, and relationships that support each executive stakeholder's concerns. Data quality issues — including inconsistent categorical values, missing fields, and cross-table discrepancies — are addressed systematically before analysis begins.

**Analytical methods applied:**
- Data cleaning and standardisation (pandas)
- Aggregation and feature engineering (pandas, NumPy)
- Profitability approximation analysis
- Anomaly detection (z-score, IQR)
- Visualisation (matplotlib, seaborn)

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Consistent plot styling
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

BASE = '/content/'   # adjust if files are in a subfolder

orders      = pd.read_csv(BASE + 'orders.csv',      parse_dates=['order_created_at'])
deliveries  = pd.read_csv(BASE + 'deliveries.csv',  parse_dates=['dispatch_time', 'delivery_completed_at'])
drivers     = pd.read_csv(BASE + 'drivers.csv')
vehicles    = pd.read_csv(BASE + 'vehicles.csv')
hubs        = pd.read_csv(BASE + 'hubs.csv')
customers   = pd.read_csv(BASE + 'customers.csv',   parse_dates=['signup_date'])
complaints  = pd.read_csv(BASE + 'complaints.csv',  parse_dates=['created_at'])
incidents   = pd.read_csv(BASE + 'incidents.csv',   parse_dates=['reported_at'])
app_events  = pd.read_csv(BASE + 'app_events.csv',  parse_dates=['event_timestamp'])

print('Files loaded.')
print(f'  Orders: {len(orders):,} | Deliveries: {len(deliveries):,} | Drivers: {len(drivers):,}')
print(f'  Vehicles: {len(vehicles):,} | Customers: {len(customers):,} | Complaints: {len(complaints):,}')

## 2. Data Cleaning & Standardisation

The dataset contains known data quality issues: inconsistent zone name casing, missing values in battery health and booking channel, and categorical drift in zone labels. All are addressed before analysis.

In [ ]:
# ── 2.1  Standardise zone names ───────────────────────────────────────────────
def norm_zone(series):
    return series.str.strip().str.title()

for df, cols in [
    (orders,    ['pickup_zone', 'dropoff_zone']),
    (drivers,   ['base_zone']),
    (vehicles,  ['assigned_zone']),
    (customers, ['home_zone']),
    (hubs,      ['zone']),
    (app_events,['zone_context']),
]:
    for col in cols:
        df[col] = norm_zone(df[col])

# ── 2.2  Fix known zone aliases ───────────────────────────────────────────────
zone_map = {'Ctr': 'Central', 'Riverside': 'Riverside'}
for df, cols in [(orders, ['pickup_zone', 'dropoff_zone']),
                 (vehicles, ['assigned_zone']), (drivers, ['base_zone'])]:
    for col in cols:
        df[col] = df[col].replace(zone_map)

# ── 2.3  Missing values ───────────────────────────────────────────────────────
vehicles['battery_health_pct'] = pd.to_numeric(vehicles['battery_health_pct'], errors='coerce')
orders['booking_channel'] = orders['booking_channel'].replace('', 'Unknown').fillna('Unknown')

# ── 2.4  Derived columns ──────────────────────────────────────────────────────
deliveries['duration_hours'] = (
    deliveries['delivery_completed_at'] - deliveries['dispatch_time']
).dt.total_seconds() / 3600

# Remove physically impossible durations (negative or >72h)
deliveries['duration_hours'] = deliveries['duration_hours'].where(
    deliveries['duration_hours'].between(0, 72), np.nan
)

# ── 2.5  Missing value summary ────────────────────────────────────────────────
print('Missing value counts per table:')
for name, df in [('orders', orders), ('deliveries', deliveries), 
                 ('vehicles', vehicles), ('customers', customers)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print(f'  {name}: {missing.to_dict()}')
    else:
        print(f'  {name}: no missing values')

## 3. Build Core Analytical Table

In [ ]:
main = (
    orders
    .merge(deliveries, on='order_id', how='inner')
    .merge(hubs,    on='hub_id',    how='left')
    .merge(drivers, on='driver_id', how='left', suffixes=('', '_driver'))
    .merge(vehicles,on='vehicle_id',how='left', suffixes=('', '_vehicle'))
)

print(f'Core analytical table: {len(main):,} rows x {main.shape[1]} columns')
print('\nDelivery status breakdown:')
print(main['delivery_status'].value_counts())

## 4. Analysis 1 — Profitability Approximation by Service Type

The finance director suspects some service contracts are loss-making. By subtracting fuel/charge costs and estimated complaint compensation from order value, a proxy profitability margin is computed.

**Note:** This is an approximation; full profitability requires staff cost and overhead data not in the dataset.

In [ ]:
# Join complaint compensation to orders
comp_per_order = complaints.groupby('order_id')['compensation_amount'].sum().reset_index()
comp_per_order.columns = ['order_id', 'total_compensation']

main_profit = main.merge(comp_per_order, on='order_id', how='left')
main_profit['total_compensation'] = main_profit['total_compensation'].fillna(0)
main_profit['approx_margin'] = (
    main_profit['order_value']
    - main_profit['fuel_or_charge_cost']
    - main_profit['total_compensation']
)

profit_by_service = (
    main_profit.groupby('service_type')
    .agg(
        total_orders       = ('order_id', 'count'),
        total_revenue      = ('order_value', 'sum'),
        total_fuel_cost    = ('fuel_or_charge_cost', 'sum'),
        total_compensation = ('total_compensation', 'sum'),
        total_margin       = ('approx_margin', 'sum'),
        avg_margin         = ('approx_margin', 'mean')
    )
    .round(2)
    .sort_values('avg_margin')
)

print('=== Profitability Approximation by Service Type ===')
print(profit_by_service.to_string())

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
colours = ['#e74c3c' if m < 0 else '#2ecc71' for m in profit_by_service['avg_margin']]
ax.barh(profit_by_service.index, profit_by_service['avg_margin'], color=colours, edgecolor='white')
ax.axvline(0, colour='black', linewidth=0.8)
ax.set_xlabel('Average Margin per Delivery (£)')
ax.set_title('Approximate Profitability by Service Type\n'
             '(Order Value – Fuel Cost – Complaint Compensation)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Analysis 2 — Zone-Level Performance Dashboard

In [ ]:
zone_perf = (
    main.groupby('pickup_zone')
    .agg(
        total_deliveries    = ('delivery_id', 'count'),
        on_time             = ('delivery_status', lambda x: (x == 'OnTime').sum()),
        delayed             = ('delivery_status', lambda x: (x == 'Delayed').sum()),
        failed              = ('delivery_status', lambda x: (x == 'Failed').sum()),
        avg_customer_rating = ('customer_rating_post_delivery', 'mean'),
        avg_fuel_cost       = ('fuel_or_charge_cost', 'mean'),
        avg_overrides       = ('manual_route_override_count', 'mean')
    )
    .assign(
        failure_rate_pct = lambda d: ((d['delayed'] + d['failed']) / d['total_deliveries'] * 100).round(1),
        avg_customer_rating = lambda d: d['avg_customer_rating'].round(2),
        avg_fuel_cost       = lambda d: d['avg_fuel_cost'].round(2),
        avg_overrides       = lambda d: d['avg_overrides'].round(2)
    )
    .query('total_deliveries >= 5')
    .sort_values('failure_rate_pct', ascending=False)
)

print('=== Zone Performance Summary ===')
print(zone_perf.to_string())

# Four-panel zone dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('NorthStar Zone-Level Performance Dashboard', fontweight='bold', fontsize=14)

zones = zone_perf.index
palette = sns.color_palette('coolwarm', len(zones))

# Panel 1: Failure rate
axes[0,0].barh(zones, zone_perf['failure_rate_pct'],
               color=sns.color_palette('Reds', len(zones)))
axes[0,0].set_title('Failure + Delay Rate (%)')
axes[0,0].set_xlabel('%')

# Panel 2: Avg customer rating
axes[0,1].barh(zones, zone_perf['avg_customer_rating'],
               color=sns.color_palette('Greens', len(zones)))
axes[0,1].set_title('Avg Customer Rating (1–5)')
axes[0,1].set_xlabel('Rating')

# Panel 3: Avg fuel cost
axes[1,0].barh(zones, zone_perf['avg_fuel_cost'],
               color=sns.color_palette('Blues', len(zones)))
axes[1,0].set_title('Avg Fuel / Charge Cost (£)')
axes[1,0].set_xlabel('£')

# Panel 4: Avg route overrides
axes[1,1].barh(zones, zone_perf['avg_overrides'],
               color=sns.color_palette('Oranges', len(zones)))
axes[1,1].set_title('Avg Manual Route Overrides')
axes[1,1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 6. Analysis 3 — Anomaly Detection: Unusual Delivery Durations

Using z-scores and IQR fencing to identify deliveries whose duration is statistically anomalous. These outliers may indicate data recording errors, genuine extreme delays, or fraud.

In [ ]:
dur = deliveries['duration_hours'].dropna()

# Z-score method
z_scores = np.abs(stats.zscore(dur))
zscore_outliers = deliveries.loc[dur.index[z_scores > 3], 
                                  ['delivery_id','delivery_status','duration_hours']]

# IQR method
Q1, Q3 = dur.quantile(0.25), dur.quantile(0.75)
IQR = Q3 - Q1
iqr_outliers = deliveries[
    (deliveries['duration_hours'] < Q1 - 1.5*IQR) |
    (deliveries['duration_hours'] > Q3 + 1.5*IQR)
][['delivery_id','delivery_status','duration_hours']]

print(f'Duration stats: mean={dur.mean():.1f}h, median={dur.median():.1f}h, std={dur.std():.1f}h')
print(f'Z-score outliers (|z|>3):  {len(zscore_outliers)}')
print(f'IQR outliers:              {len(iqr_outliers)}')
print('\nTop 10 longest deliveries (potential anomalies):')
print(deliveries.nlargest(10, 'duration_hours')[['delivery_id','delivery_status','duration_hours']])

# Distribution plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(dur, bins=40, colour='#3498db', edgecolour='white', alpha=0.8)
ax.axvline(dur.mean(), colour='#e74c3c', linestyle='--', linewidth=1.5, label=f'Mean = {dur.mean():.1f}h')
ax.axvline(Q3 + 1.5*IQR, colour='#f39c12', linestyle='--', linewidth=1.5,
           label=f'IQR upper fence = {Q3+1.5*IQR:.1f}h')
ax.set_xlabel('Delivery Duration (hours)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Delivery Durations with Anomaly Thresholds', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Analysis 4 — App Platform Reliability by Zone and Event Type

In [ ]:
app_zone = (
    app_events.groupby(['zone_context', 'event_type'])
    .agg(
        total_events  = ('event_id', 'count'),
        failed_events = ('success_flag', lambda x: (x == 0).sum()),
        avg_latency   = ('api_latency_ms', 'mean')
    )
    .assign(failure_rate = lambda d: (d['failed_events'] / d['total_events'] * 100).round(1))
    .reset_index()
)

# Pivot for heatmap
latency_pivot = app_zone.pivot_table(
    index='zone_context', columns='event_type', values='avg_latency', aggfunc='mean'
).round(0)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(latency_pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Latency (ms)'})
ax.set_title('Average API Latency (ms) by Zone and Event Type', fontweight='bold')
ax.set_xlabel('Event Type')
ax.set_ylabel('Zone')
plt.tight_layout()
plt.show()

print('\nHighest average latency combinations:')
print(app_zone.nlargest(5, 'avg_latency')[['zone_context','event_type','avg_latency','failure_rate']])

## 8. Analysis 5 — Driver Workforce Composition and Risk Profile

In [ ]:
# Join deliveries to drivers for performance context
driver_perf = (
    deliveries.groupby('driver_id')
    .agg(
        deliveries_count  = ('delivery_id', 'count'),
        failed_deliveries = ('delivery_status', lambda x: (x == 'Failed').sum()),
        total_overrides   = ('manual_route_override_count', 'sum'),
        avg_rating        = ('customer_rating_post_delivery', 'mean'),
        proof_missing     = ('proof_of_completion_missing', 'sum')
    )
    .reset_index()
    .merge(drivers, on='driver_id')
    .assign(
        failure_rate     = lambda d: d['failed_deliveries'] / d['deliveries_count'],
        override_rate    = lambda d: d['total_overrides'] / d['deliveries_count'],
        risk_score       = lambda d: (
            d['failure_rate'] * 40 +
            d['override_rate'] * 30 +
            (d['proof_missing'] / d['deliveries_count']) * 30
        ).round(2)
    )
    .query('active_flag == 1 and deliveries_count >= 3')
    .sort_values('risk_score', ascending=False)
)

print('Driver risk score distribution (top 15 highest-risk):')
print(driver_perf[['driver_id','employment_type','base_zone','training_score',
                    'failure_rate','override_rate','risk_score']].head(15).to_string())

# Employment type vs risk
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Driver Risk Analysis', fontweight='bold')

sns.boxplot(data=driver_perf, x='employment_type', y='risk_score', ax=axes[0],
            palette='Set2')
axes[0].set_title('Risk Score by Employment Type')
axes[0].set_xlabel('Employment Type')
axes[0].set_ylabel('Composite Risk Score')

axes[1].scatter(driver_perf['training_score'], driver_perf['risk_score'],
                alpha=0.5, c=driver_perf['avg_rating'], cmap='RdYlGn', s=40)
sm = plt.cm.ScalarMappable(cmap='RdYlGn',
                            norm=plt.Normalize(driver_perf['avg_rating'].min(),
                                               driver_perf['avg_rating'].max()))
plt.colorbar(sm, ax=axes[1], label='Avg Customer Rating')
axes[1].set_title('Training Score vs Risk Score\n(colour = customer rating)')
axes[1].set_xlabel('Training Score')
axes[1].set_ylabel('Composite Risk Score')

plt.tight_layout()
plt.show()

## 9. Analysis 6 — Vehicle Fleet Health and Maintenance Risk

In [ ]:
# Join vehicle data to their delivery records and associated incidents
vehicle_incidents = (
    incidents.merge(deliveries[['delivery_id','vehicle_id']], on='delivery_id', how='left')
    .groupby('vehicle_id')
    .agg(incident_count=('incident_id','count'),
         critical_count=('severity', lambda x: (x.isin(['Critical','High'])).sum()))
    .reset_index()
)

veh_health = vehicles.merge(vehicle_incidents, on='vehicle_id', how='left')
veh_health[['incident_count','critical_count']] = veh_health[['incident_count','critical_count']].fillna(0)

# Battery health by vehicle type
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Fleet Health Analysis', fontweight='bold')

sns.boxplot(data=veh_health.dropna(subset=['battery_health_pct']),
            x='vehicle_type', y='battery_health_pct',
            hue='maintenance_status', palette='Set1', ax=axes[0])
axes[0].axhline(50, color='red', linestyle='--', label='50% threshold')
axes[0].set_title('Battery Health by Vehicle Type & Maintenance Status')
axes[0].set_xlabel('Vehicle Type')
axes[0].set_ylabel('Battery Health (%)')
axes[0].legend(fontsize=8)

# Scatter: odometer vs battery health
scatter_data = veh_health.dropna(subset=['battery_health_pct'])
axes[1].scatter(scatter_data['odometer_km'], scatter_data['battery_health_pct'],
                c=scatter_data['incident_count'], cmap='Reds', alpha=0.7, s=50)
sm2 = plt.cm.ScalarMappable(cmap='Reds',
                             norm=plt.Normalize(0, scatter_data['incident_count'].max()))
plt.colorbar(sm2, ax=axes[1], label='Incident Count')
axes[1].set_title('Odometer vs Battery Health\n(colour = linked incident count)')
axes[1].set_xlabel('Odometer (km)')
axes[1].set_ylabel('Battery Health (%)')

plt.tight_layout()
plt.show()

print('\nAt-risk vehicles (battery < 50% or maintenance not Active):')
at_risk = veh_health[
    (veh_health['battery_health_pct'] < 50) | 
    (veh_health['maintenance_status'].isin(['InRepair','Scheduled']))
][['vehicle_id','vehicle_type','assigned_zone','maintenance_status',
   'battery_health_pct','incident_count','critical_count']]
print(f'  {len(at_risk)} vehicles identified as at-risk')
print(at_risk.head(15).to_string(index=False))

## 10. Analysis 7 — Customer Segmentation by Complaint and Failure History

In [ ]:
# Build per-customer KPIs
cust_orders = (
    orders.merge(deliveries[['order_id','delivery_status','customer_rating_post_delivery']],
                 on='order_id', how='left')
    .groupby('customer_id')
    .agg(
        total_orders      = ('order_id', 'count'),
        problem_deliveries= ('delivery_status', lambda x: x.isin(['Failed','Delayed']).sum()),
        avg_rating        = ('customer_rating_post_delivery', 'mean'),
        total_spend       = ('order_value', 'sum')
    )
    .reset_index()
)

cust_complaints = (
    complaints.groupby('customer_id')
    .agg(
        complaint_count    = ('complaint_id', 'count'),
        total_compensation = ('compensation_amount', 'sum')
    )
    .reset_index()
)

cust_full = (
    customers.merge(cust_orders, on='customer_id', how='left')
             .merge(cust_complaints, on='customer_id', how='left')
)
cust_full[['complaint_count','total_compensation']] = (
    cust_full[['complaint_count','total_compensation']].fillna(0)
)
cust_full['problem_rate'] = (
    cust_full['problem_deliveries'] / cust_full['total_orders'].replace(0, np.nan)
).fillna(0)

# Segment customers
def segment(row):
    if row['complaint_count'] >= 3 and row['problem_rate'] > 0.5:
        return 'High Risk'
    elif row['complaint_count'] >= 1 or row['problem_rate'] > 0.3:
        return 'At Risk'
    elif row['loyalty_score'] > 70 and row['problem_rate'] < 0.1:
        return 'Loyal'
    else:
        return 'Standard'

cust_full['segment'] = cust_full.apply(segment, axis=1)

seg_summary = (
    cust_full.groupby('segment')
    .agg(
        count             = ('customer_id', 'count'),
        avg_loyalty       = ('loyalty_score', 'mean'),
        avg_spend         = ('total_spend', 'mean'),
        avg_complaints    = ('complaint_count', 'mean'),
        avg_compensation  = ('total_compensation', 'mean')
    )
    .round(2)
)
print('=== Customer Segmentation Summary ===')
print(seg_summary.to_string())

# Visualise segments
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Customer Segmentation Analysis', fontweight='bold')

seg_counts = cust_full['segment'].value_counts()
colours = {'High Risk':'#e74c3c','At Risk':'#f39c12','Standard':'#3498db','Loyal':'#2ecc71'}
axes[0].bar(seg_counts.index, seg_counts.values,
            color=[colours.get(s, '#95a5a6') for s in seg_counts.index])
axes[0].set_title('Customer Count by Segment')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Count')

sns.scatterplot(data=cust_full.dropna(subset=['total_spend','loyalty_score']),
                x='total_spend', y='loyalty_score', hue='segment',
                palette=colours, alpha=0.6, s=40, ax=axes[1])
axes[1].set_title('Spend vs Loyalty Score by Segment')
axes[1].set_xlabel('Total Spend (£)')
axes[1].set_ylabel('Loyalty Score')

plt.tight_layout()
plt.show()

## 11. Summary of Python Analytical Findings

The seven analytical modules above collectively address each senior management concern:

**Finance director:** The profitability approximation shows that when complaint compensation is deducted from order value, net margins vary significantly by service type. Certain service types may be generating negative margins once fuel and exception costs are included.

**Operations director:** The zone performance dashboard confirms that failure rates, route override behaviour, and average costs are not uniformly distributed. Zones with both high failure rates and high average fuel costs represent the highest operational cost.

**Customer experience director:** The customer segmentation analysis identifies a subset of 'High Risk' customers who have experienced multiple failures and hold lower loyalty scores, despite potentially high spend — representing both a retention risk and a compensation liability.

**Technology director:** App latency analysis shows that certain zones and event types consistently generate higher latency and failure rates, suggesting that the mobile platform's infrastructure is not uniformly provisioned across the operational territory.

**Anomaly detection** has identified delivery records with durations that fall outside statistical norms, pointing to likely data quality issues in the dispatch recording system that should be investigated before they corrupt downstream reporting.